# 01 — Exploração da API Pública do Datajud

Este notebook demonstra como:
- Autenticar e fazer requisições à API
- Explorar a estrutura dos dados retornados
- Entender os campos disponíveis para análise

**Referência:** https://www.cnj.jus.br/sistemas/datajud/api-publica/

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
from datajud import DatajudClient

## 1. Inicializar o cliente

In [ ]:
client = DatajudClient()
print('Cliente inicializado com sucesso.')

## 2. Busca simples — processos do TJSP

In [ ]:
resultado = client.buscar(
    tribunal='tjsp',
    tamanho=5,
)

print(f"Total de processos no índice: {resultado['total']:,}")
print(f"Processos retornados: {len(resultado['processos'])}")

## 3. Inspecionar estrutura de um processo

In [ ]:
if resultado['processos']:
    p = resultado['processos'][0]
    print(f'Número: {p.numero_processo}')
    print(f'Tribunal: {p.tribunal}')
    print(f'Grau: {p.grau}')
    print(f'Classe: {p.classe_nome}')
    print(f'Assuntos: {p.assuntos_nomes}')
    print(f'Órgão Julgador: {p.orgao_julgador_nome}')
    print(f'Data Ajuizamento: {p.data_ajuizamento}')
    print(f'Total de movimentos: {len(p.movimentos)}')
    
    if p.movimentos:
        print('\nPrimeiro movimento:')
        m = p.movimentos[0]
        print(f'  Código: {m.codigo}')
        print(f'  Nome: {m.nome}')
        print(f'  Data: {m.data_hora}')

## 4. Busca por classe processual

Exemplo: classe 436 = **Procedimento Comum** (muito frequente no TJSP)

In [ ]:
# Exemplo: Procedimento Comum Cível (código 436)
resultado_classe = client.buscar(
    tribunal='tjsp',
    classe_codigo=436,
    tamanho=10,
    data_inicio='2023-01-01',
    data_fim='2023-12-31',
)

print(f"Total de Procedimentos Comuns em 2023 (TJSP): {resultado_classe['total']:,}")

## 5. Query Elasticsearch bruta (avançado)

In [ ]:
# Aggregation: top assuntos no TJSP
query_agg = {
    'size': 0,
    'aggs': {
        'top_assuntos': {
            'terms': {
                'field': 'assuntos.nome.keyword',
                'size': 15
            }
        }
    }
}

resp = client.buscar_query_raw('tjsp', query_agg)
buckets = resp.get('aggregations', {}).get('top_assuntos', {}).get('buckets', [])

df_assuntos = pd.DataFrame(buckets).rename(columns={'key': 'assunto', 'doc_count': 'total'})
print(df_assuntos.to_string(index=False))

## 6. Comparar volume entre tribunais

In [ ]:
import matplotlib.pyplot as plt

tribunais = ['tjsp', 'tjrj', 'tjmg', 'tjrs', 'tjpr']
totais = {}

for t in tribunais:
    try:
        total = client.contar(t)
        totais[t.upper()] = total
        print(f'{t.upper()}: {total:,} processos')
    except Exception as e:
        print(f'{t.upper()}: erro — {e}')

if totais:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(totais.keys(), totais.values(), color='steelblue')
    ax.set_title('Volume de processos por tribunal')
    ax.set_ylabel('Total de processos')
    plt.tight_layout()
    plt.show()